In [ ]:
# W3D3: Decision Trees & Random Forests — run this single cell from top to bottom.
from pathlib import Path
import json
import warnings
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

RANDOM_STATE = 42

# Jupyter has no __file__; find the repository from the kernel working directory.
cwd = Path.cwd().resolve()
root = next((folder for folder in (cwd, *cwd.parents) if (folder / 'week3_Notebooks').is_dir()), cwd)
output_dir = root / 'outputs' / 'w3d3_decision_trees'
output_dir.mkdir(parents=True, exist_ok=True)

def gini_impurity(counts):
    total = sum(counts)
    return 0.0 if total == 0 else 1 - sum((count / total) ** 2 for count in counts)

def information_gain(parent, left, right):
    total = sum(parent)
    return 0.0 if total == 0 else gini_impurity(parent) - (sum(left) / total * gini_impurity(left) + sum(right) / total * gini_impurity(right))

def evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_test)[:, 1]
    return {
        'train_accuracy': accuracy_score(y_train, model.predict(X_train)),
        'test_accuracy': accuracy_score(y_test, model.predict(X_test)),
        'test_f1': f1_score(y_test, model.predict(X_test)),
        'test_roc_auc': roc_auc_score(y_test, probability),
    }

# Stratification preserves the malignant/benign balance in both splits.
X, y = load_breast_cancer(return_X_y=True)
feature_names = [f'feature_{index}' for index in range(X.shape[1])]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# An unrestricted tree exposes overfitting; CV selects safer tree complexity.
baseline_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
baseline_metrics = evaluate(baseline_tree, X_train, X_test, y_train, y_test)
tree_search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    {'criterion': ['gini', 'entropy'], 'max_depth': [2, 3, 4, 5, 6], 'min_samples_leaf': [1, 3, 5, 10]},
    scoring='roc_auc', cv=5, n_jobs=1
)
tree_search.fit(X_train, y_train)
tuned_tree = tree_search.best_estimator_
tuned_tree_metrics = evaluate(tuned_tree, X_train, X_test, y_train, y_test)

forest_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    {'n_estimators': [100, 200], 'max_depth': [None, 4, 6], 'min_samples_leaf': [1, 3]},
    scoring='roc_auc', cv=5, n_jobs=1
)
forest_search.fit(X_train, y_train)
forest = forest_search.best_estimator_
forest_metrics = evaluate(forest, X_train, X_test, y_train, y_test)

# Visualise the selected tree and the forest's most useful features.
figure, axis = plt.subplots(figsize=(18, 9))
plot_tree(tuned_tree, feature_names=feature_names, class_names=['malignant', 'benign'], filled=True, rounded=True, ax=axis)
figure.tight_layout()
tree_plot = output_dir / 'tuned_tree.png'
figure.savefig(tree_plot, dpi=160)
plt.close(figure)

importance = pd.Series(forest.feature_importances_, index=feature_names).nlargest(10).sort_values()
figure, axis = plt.subplots(figsize=(8, 5))
importance.plot.barh(ax=axis, color='#3976af', title='Random Forest: top 10 feature importances')
axis.set_xlabel('Mean impurity decrease')
figure.tight_layout()
importance_plot = output_dir / 'forest_feature_importance.png'
figure.savefig(importance_plot, dpi=160)
plt.close(figure)

cv_results = pd.DataFrame(tree_search.cv_results_)[['params', 'mean_test_score', 'rank_test_score']].sort_values('rank_test_score')
cv_path = output_dir / 'tree_cv_results.csv'
cv_results.to_csv(cv_path, index=False)
summary = {
    'baseline_tree': baseline_metrics,
    'tuned_tree': {**tuned_tree_metrics, 'best_cv_roc_auc': tree_search.best_score_, 'parameters': tree_search.best_params_},
    'random_forest': {**forest_metrics, 'best_cv_roc_auc': forest_search.best_score_, 'parameters': forest_search.best_params_},
    'overfit_gap_reduction': (baseline_metrics['train_accuracy'] - baseline_metrics['test_accuracy']) - (tuned_tree_metrics['train_accuracy'] - tuned_tree_metrics['test_accuracy']),
}
metrics_path = output_dir / 'metrics.json'
metrics_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

# MLflow is optional: model results and local artifacts remain available if its kernel state is unhealthy.
try:
    import mlflow
    mlflow.set_tracking_uri((output_dir / 'mlruns').resolve().as_uri())
    mlflow.set_experiment('w3d3_decision_trees')
    with mlflow.start_run(run_name='tree-and-forest-tuning'):
        mlflow.log_params({f'tree_{key}': value for key, value in tree_search.best_params_.items()})
        mlflow.log_params({f'forest_{key}': value for key, value in forest_search.best_params_.items()})
        mlflow.log_metrics({f'baseline_{key}': value for key, value in baseline_metrics.items()})
        mlflow.log_metrics({f'tuned_tree_{key}': value for key, value in tuned_tree_metrics.items()})
        mlflow.log_metrics({f'forest_{key}': value for key, value in forest_metrics.items()})
        for artifact in (tree_plot, importance_plot, cv_path, metrics_path):
            mlflow.log_artifact(str(artifact), artifact_path='evidence')
except (AttributeError, ImportError) as error:
    warnings.warn(f'MLflow logging skipped: {error}. Restart the kernel to enable it.', RuntimeWarning)

print('Gini([5, 5]):', gini_impurity([5, 5]))
print('Information gain from pure split:', information_gain([5, 5], [5, 0], [0, 5]))
print('Saved evidence to:', output_dir)
summary